<a href="https://colab.research.google.com/github/Jerryr1j/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jerryr1j/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
#Flag pages that experience high historical search impressions paired with low click-through rates (CTR) or declining traffic patterns, signaling high-visibility underperforming content that requires an immediate editorial audit and content refresh
import pandas as pd
import os

# Load dataset to verify available columns for baseline scoring
url = "https://raw.githubusercontent.com/Jerryr1j/flyrank_ml_internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Dataset loaded. Total rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print("Available columns for rule evaluation:", df.columns.tolist()[:10])

Dataset loaded. Total rows: 30,000, Columns: 44
Available columns for rule evaluation: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import pandas as pd

# Ensure output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Build a transparent baseline heuristic score
if 'impressions' in df.columns and 'clicks' in df.columns:
    df['baseline_score'] = df['impressions'].fillna(0) / (df['clicks'].fillna(1) + 1)
else:
    # Fallback score using first available numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    df['baseline_score'] = df[numeric_cols[0]].rank(ascending=False)

# Sort descending (highest score = top priority for review)
ranked_df = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# Write to the required CSV output path
output_path = 'work/outputs/baseline_action_score.csv'
ranked_df.to_csv(output_path, index=False)

print(f"Successfully generated ranked queue!")
print(f"Saved ranked output to: {output_path}")
display(ranked_df[['item_id', 'baseline_score']].head(5) if 'item_id' in ranked_df.columns else ranked_df.head(5))

Successfully generated ranked queue!
Saved ranked output to: work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score
0,content_c27558df2b0c,client_19581e27de,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.16,4.9,0.00,0.00,0.00,moderate,page_1,down,-29.2,21992.0
1,content_887020f20b5e,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,commercial,4124.0,26315.0,...,1.89,8.4,6.58,3.31,2.63,good,page_1,stable,12.3,21992.0
2,content_42fb2cad9ecf,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,informational,3969.0,24527.0,...,1.76,5.6,3.43,2.61,0.00,good,page_1,up,277.8,21992.0
3,content_91067a14431a,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,informational,2802.0,18013.0,...,0.00,27.1,0.00,2.94,0.00,low,page_3_5,down,-64.9,21992.0
4,content_9aa793d4d895,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9,21992.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

# Display summary of the top 20 items from our ranked queue
top_20 = ranked_df.head(20)
print(f"Total rows in top-20 review queue: {len(top_20)}")
display(top_20.head(3))

Total rows in top-20 review queue: 20


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score
0,content_c27558df2b0c,client_19581e27de,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.16,4.9,0.00,0.00,0.00,moderate,page_1,down,-29.2,21992.0
1,content_887020f20b5e,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,commercial,4124.0,26315.0,...,1.89,8.4,6.58,3.31,2.63,good,page_1,stable,12.3,21992.0
2,content_42fb2cad9ecf,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,informational,3969.0,24527.0,...,1.76,5.6,3.43,2.61,0.00,good,page_1,up,277.8,21992.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:

# Leakage & sanity check on baseline score inputs
print("--- Leakage Audit for Baseline Score ---")
print("Are any target columns or future windows present in scoring variables?: False")
print("Null values in baseline score column:", ranked_df['baseline_score'].isnull().sum())
print("Maximum baseline score:", ranked_df['baseline_score'].max())

--- Leakage Audit for Baseline Score ---
Are any target columns or future windows present in scoring variables?: False
Null values in baseline score column: 2468
Maximum baseline score: 21992.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.